In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# ─────────────────────────────────────────
# 1. DEVICE SETUP
# ─────────────────────────────────────────
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
# ─────────────────────────────────────────
# 2. TRANSFORMATIONS
# ─────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize RGB channels
])

In [ ]:
# ─────────────────────────────────────────
# 3. DOWNLOAD DATASET & PREPARE DATALOADER
# ─────────────────────────────────────────
training_data = datasets.CIFAR10(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.CIFAR10(
    root="data",
    train=False,
    download=True,
    transform=transform
)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader  = DataLoader(test_data,     batch_size=64, shuffle=False)

# CIFAR-10 class names
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# ─────────────────────────────────────────
# 4. DISPLAY A SAMPLE IMAGE
# ─────────────────────────────────────────
data_iter = iter(train_dataloader)
images, labels = next(data_iter)

# un-normalize for display
img = images[0] / 2 + 0.5
npimg = img.numpy()
plt.imshow(np.transpose(npimg, (1, 2, 0)))
plt.title(f"Sample Image: {classes[labels[0]]}")
plt.axis('off')
plt.show()

In [ ]:
# ─────────────────────────────────────────
# 5. NEURAL NETWORK (5 hidden layers)
# ─────────────────────────────────────────
# CIFAR-10 images are 32x32x3 → input size = 3072
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(32*32*3, 1024),   # Hidden Layer 1
            nn.ReLU(),
            nn.Linear(1024, 512),        # Hidden Layer 2
            nn.ReLU(),
            nn.Linear(512, 256),         # Hidden Layer 3
            nn.ReLU(),
            nn.Linear(256, 128),         # Hidden Layer 4
            nn.ReLU(),
            nn.Linear(128, 64),          # Hidden Layer 5
            nn.ReLU(),
            nn.Linear(64, 10),           # Output Layer (10 classes)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(f"\nModel Structure:\n{model}\n")



In [ ]:
# ─────────────────────────────────────────
# 6. LOSS FUNCTION & OPTIMIZER
# ─────────────────────────────────────────
loss_fn   = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [ ]:
# ─────────────────────────────────────────
# 7. TRAIN & TEST LOOPS
# ─────────────────────────────────────────
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    size = len(dataloader.dataset)
    total_loss = 0
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        if batch % 100 == 0:
            current = batch * 64 + len(X)
            print(f"  loss: {loss.item():>7f}  [{current:>5d}/{size:>5d}]")

    avg_loss = total_loss / len(dataloader)
    return avg_loss


def test_loop(dataloader, model, loss_fn):
    model.eval()
    size        = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred       = model(X)
            test_loss += loss_fn(pred, y).item()
            correct   += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    accuracy   = (100 * correct) / size
    print(f"  Test  → Accuracy: {accuracy:>0.2f}%, Avg Loss: {test_loss:>8f}")
    return accuracy, test_loss

In [ ]:
# ─────────────────────────────────────────
# 8. TRAIN FOR 30, 50, AND 100 EPOCHS
# ─────────────────────────────────────────
results = {}

for num_epochs in [30, 50, 100]:
    print(f"\n{'='*55}")
    print(f"  Training for {num_epochs} EPOCHS")
    print(f"{'='*55}")

    # Re-initialize model & optimizer fresh for each run
    model     = NeuralNetwork().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    for t in range(num_epochs):
        print(f"\nEpoch {t+1}/{num_epochs}\n{'-'*30}")
        train_loss           = train_loop(train_dataloader, model, loss_fn, optimizer)
        accuracy, test_loss  = test_loop(test_dataloader,  model, loss_fn)

    # Record final epoch results
    results[num_epochs] = {
        "train_loss": train_loss,
        "test_loss":  test_loss,
        "accuracy":   accuracy
    }

In [ ]:
# ─────────────────────────────────────────
# 8. TRAIN FOR 30, 50, AND 100 EPOCHS
# ─────────────────────────────────────────
results = {}

for num_epochs in [30, 50, 100]:
    print(f"\n{'='*55}")
    print(f"  Training for {num_epochs} EPOCHS")
    print(f"{'='*55}")

    # Re-initialize model & optimizer fresh for each run
    model     = NeuralNetwork().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

    epoch_history = []  # track per-epoch stats

    for t in range(num_epochs):
        print(f"\nEpoch {t+1}/{num_epochs}\n{'-'*30}")
        train_loss          = train_loop(train_dataloader, model, loss_fn, optimizer)
        accuracy, test_loss = test_loop(test_dataloader,  model, loss_fn)

        epoch_history.append({
            "epoch":      t + 1,
            "train_loss": train_loss,
            "test_loss":  test_loss,
            "accuracy":   accuracy
        })

    results[num_epochs] = epoch_history

In [ ]:
# ─────────────────────────────────────────
# 9. PRINT SUMMARY RESULTS
# ─────────────────────────────────────────
for num_epochs, history in results.items():
    print(f"\n{'='*55}")
    print(f"  RESULTS — CIFAR-10 | {num_epochs} EPOCHS")
    print(f"{'='*55}")
    print(f"{'Epoch':<10} {'Train Loss':<15} {'Test Loss':<15} {'Accuracy':<10}")
    print(f"{'-'*50}")
    for row in history:
        print(f"{row['epoch']:<10} {row['train_loss']:<15.6f} {row['test_loss']:<15.6f} {row['accuracy']:<10.2f}%")
    print(f"{'-'*50}")
    best = max(history, key=lambda x: x['accuracy'])
    print(f"  Best Accuracy: {best['accuracy']:.2f}% at Epoch {best['epoch']}")

print("\nDone!")